<a href="https://colab.research.google.com/github/Logan-Das/MMA3001_Project/blob/main/porkRasherPackaging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q roboflow ultralytics
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 441.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 9.1 MB/s eta 0:00:00
Wed Oct 14 00:41:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                     

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging")
version = project.version(4)
dataset = version.download("yolov11")

print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pork-Rasher-Error(Packaging)-4 in yolov11:: 100%|██████████| 7110/7110 [00:01<00:00, 5718.22it/s]

/content/Pork-Rasher-Error(Packaging)-4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/pork_rasher",
    name="yolo11n_baseline",
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pork-Rasher-Error(Packaging)-4/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchs

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79cd41e53380>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [7]:
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/pork_rasher/yolo11n_baseline/weights/best.pt")

# True test-split metrics
m = model.val(data=f"{dataset.location}/data.yaml", split="test")
for i, name in model.names.items():
    print(name, "AP50:", m.box.ap50[i] if i in m.box.ap_class_index else "no instances")

# Tray-level pass/fail: fail if any detection above threshold
import glob, os
test_imgs = glob.glob(f"{dataset.location}/test/images/*")
for t in [0.1, 0.25, 0.4, 0.5]:
    flagged = sum(len(model.predict(p, conf=t, verbose=False)[0].boxes) > 0 for p in test_imgs)
    print(f"conf>={t}: {flagged}/{len(test_imgs)} trays flagged as fail")

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1518.1±416.4 MB/s, size: 66.3 KB)
val: Scanning /content/Pork-Rasher-Error(Packaging)-4/test/labels... 80 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 644.8it/s 0.1s
val: New cache created: /content/Pork-Rasher-Error(Packaging)-4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.5it/s 3.3s
                   all         80         79      0.514      0.422      0.492      0.227
            loose-meat          2          3      0.713      0.333      0.645      0.227
       packaging-error         16         23      0.534      0.478      0.457      0.129
          twisted-meat          4          4          0          0     0.0245     0.0101
              unsealed         49       

In [8]:
import glob, os
import numpy as np
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/pork_rasher/yolo11n_baseline/weights/best.pt")
imgs = sorted(glob.glob(f"{dataset.location}/test/images/*"))

# Ground truth: tray is defective if its label file has any boxes
def is_defective(img):
    lbl = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    return os.path.exists(lbl) and os.path.getsize(lbl) > 0

gt = np.array([is_defective(p) for p in imgs])
print("defective:", gt.sum(), "good:", (~gt).sum())

# Run once at a low threshold, store max confidence per image
max_conf = np.array([
    (r.boxes.conf.max().item() if len(r.boxes) else 0.0)
    for r in model.predict(imgs, conf=0.05, verbose=False, stream=True)
])

print("thr   TP  FP  FN  TN  recall  precision  false-alarm-rate  F1")
for t in [0.05, 0.1, 0.15, 0.25, 0.35, 0.5]:
    pred = max_conf >= t
    tp = (pred & gt).sum(); fp = (pred & ~gt).sum()
    fn = (~pred & gt).sum(); tn = (~pred & ~gt).sum()
    rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1)
    far = fp / max(fp + tn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    print(f"{t:<5} {tp:3d} {fp:3d} {fn:3d} {tn:3d}  {rec:.2f}    {prec:.2f}       {far:.2f}            {f1:.2f}")

defective: 68 good: 12
thr   TP  FP  FN  TN  recall  precision  false-alarm-rate  F1
0.05   64   4   4   8  0.94    0.94       0.33            0.94
0.1    64   3   4   9  0.94    0.96       0.25            0.95
0.15   64   1   4  11  0.94    0.98       0.08            0.96
0.25   62   1   6  11  0.91    0.98       0.08            0.95
0.35   60   1   8  11  0.88    0.98       0.08            0.93
0.5    42   1  26  11  0.62    0.98       0.08            0.76


In [11]:
import re, glob
import numpy as np

def frame_nums(split):
    out = []
    for p in glob.glob(f"{dataset.location}/{split}/images/*"):
        m = re.search(r"frame_with_red_(\d+)_jpg", p)
        if m: out.append(int(m.group(1)))
    return np.array(sorted(out))

train, test = frame_nums("train"), frame_nums("test")
# distance from each test frame to its nearest train frame
dist = np.array([np.abs(train - t).min() for t in test])
print("test frames:", len(test), "| train frames:", len(train))
print("median nearest-train distance:", np.median(dist))
print("test frames within 10 frames of a train frame:", (dist <= 10).sum())

test frames: 80 | train frames: 3349
median nearest-train distance: 2.5
test frames within 10 frames of a train frame: 66


In [13]:
# save everything

import os, shutil, numpy as np

save = "/content/drive/MyDrive/pork_rasher/session_backup"
os.makedirs(save, exist_ok=True)

# Test-split evaluation outputs (confusion matrix, curves)
if os.path.exists("/content/runs/detect/val"):
    shutil.copytree("/content/runs/detect/val", f"{save}/test_eval", dirs_exist_ok=True)

# Per-image max confidence and ground truth, so you can replot the pass/fail table without rerunning
np.save(f"{save}/max_conf.npy", max_conf)
np.save(f"{save}/gt.npy", gt)

# The dataset itself (version 4 as exported), so you don't depend on re-downloading
shutil.copytree(dataset.location, f"{save}/dataset_v4", dirs_exist_ok=True)

print("Saved to", save)

Saved to /content/drive/MyDrive/pork_rasher/session_backup
